# H5 → VTK Conversion Tools

Convert Pressomancy simulation HDF5 files to VTK formats for ParaView visualization.
**Run section 1 (Imports) and section 2 (Shared helpers) first**, then the writer sections you
need, and finally section 7 (Execution) to actually produce files.

Two kinds of "video" are supported by the VTKHDF writers:
- **time-sweep** — one h5 file, one frame per timestep (fixed H, varying t).
- **H-sweep** — one frame per h5 file at a fixed timestep (fixed t, varying H); the animation
  axis in ParaView becomes H. Used for the *MAE-vs-field* video (t=500, H: 0 → 30).

| Function | Output | Sweep | Bonds | Notes |
|---|---|---|---|---|
| `write_vtkhdf` | single `.vtkhdf` | time | ✓ VTK_LINE | particles + dipoles |
| `write_vtkhdf_Hsweep` | single `.vtkhdf` | H (one file/frame) | ✓ VTK_LINE | particles + dipoles |
| `write_surface_vtkhdf` | single `.vtkhdf` | time | ✓ beads | ITIM height-map mesh |
| `write_surface_vtkhdf_Hsweep` | single `.vtkhdf` | H (one file/frame) | ✓ beads | ITIM height-map mesh |
| `export_vtp_pvd` | `t_*.vtp` + `.pvd` | time | optional | other exporter |
| `write_vtk_frame_modern` | per-frame `.vtk` | manual | ✗ | legacy ASCII |

**Notebook map:** 1 Imports · 2 Shared helpers · 3 VTKHDF particle writers · 4 VTKHDF surface
writers · 5 Other exporters · 6 H5 exploration · 7 Execution.

# 1 · Imports

Run this first. All sections below rely on these imports (`detect_surface_itim` and
`calculate_bond_particles` come from `pyanal`).

In [ ]:
import h5py
import numpy as np
import os
import math
from lxml import etree
from pressomancy.analysis import H5DataSelector
from pyanal.generic_functions import min_img_dist
from pyanal.surface import detect_surface_itim
from pyanal.bond_to_parts import calculate_bond_particles

from tqdm import tqdm

# 2 · Shared helpers

Small utilities reused by every writer:
- `get_sim_paths` — build the (h5, vtk) paths for a given model/field.
- `get_bond_index_pairs` — **fast** elastomer-bond reader (via `select_particles_by_object`).
  Used by all writers; never iterate `data.particles` per-particle (that hangs on big systems).
- `get_frame_particles` — folded positions + dipoles for one timestep (HM/SM aware).

In [ ]:
from repo_paths import DATA_DIR   # repository-relative; set $MAE_DATA_DIR to read the dataset from elsewhere
import os
def get_sim_paths(A, H, seed=1, dens=0.20, K="hard", n_full_box=24000, height=20,
                  h5_root=os.path.join(DATA_DIR, "MAE-BoS/cluster/sim_data"),
                  vtk_root=os.path.join(DATA_DIR, "vtk_cluster"),
                  ext=".h5"):
    """
    Build (h5_path, vtk_dir) for MAE cluster simulations.

    A         : "HM" or "SM"
    H         : field strength
    seed      : file-level seed  (cp{A}-{seed}-H...)
    dens      : particle density
    n_full_box: number of particles in a square box bo lenght box_l[0]
    height    : height of elastomer layer
    h5_root   : base dir for HDF5 files
    vtk_root  : base dir for VTK output
    ext       : extension, e.g. ".h5" or ".h5.backup"
    """
    pdx = "pdp" if A == "HM" else "pds"
    sim_dir = f"{pdx}-{dens:.2f}_{K}-{n_full_box}_{height}"
    h5_path = os.path.join(h5_root, sim_dir, f"cp{A}-{seed}-H{H:.2f}{ext}")
    vtk_dir = os.path.join(vtk_root, sim_dir, f"cp{A}-{seed}-H{H:.2f}")
    return h5_path, vtk_dir


def get_bond_index_pairs(data, A, id_to_index):
    """
    (n_bonds, 2) int array of elastomer bond index pairs, mapped through `id_to_index`
    (a {global particle id -> local index} dict for whatever array the caller indexes).

    Fast path: select_particles_by_object(...).timestep[0].bonds, partner id at rec[4].
    Do NOT read bonds by iterating `data.particles` per-particle — that is pathologically
    slow (hangs for minutes on large systems), and the raw vlen bonds dataset is not
    readable element-wise (h5py "invalid background buffer pointer").
    """
    obj_name = "PointDipolePermanent" if A == "HM" else "PointDipoleSuperpara"
    bonded = data.select_particles_by_object(obj_name).timestep[0]
    owner_ids = np.asarray(bonded.id, dtype=np.int64)
    pairs = []
    for k, blist in enumerate(bonded.bonds):
        ai = id_to_index.get(int(owner_ids[k]))
        if ai is None:
            continue
        for rec in blist:
            ci = id_to_index.get(int(rec[4]))
            if ci is not None:
                pairs.append((ai, ci))
    return np.asarray(pairs, dtype=np.int64) if pairs else np.empty((0, 2), dtype=np.int64)


def get_frame_particles(d, A, keep_types, substrate):
    """
    Folded positions of the kept particles and their dipoles, for one timestep selection `d`.

    HM: dipoles sit on the real particles (same mask as positions).
    SM: positions are real (type 62, + optional substrate 98); dipoles come from the virtual
        dipole carriers (type 666, + optional substrate).
    """
    pds_virt_type = 666
    substrate_type = 98
    types_frame = np.asarray(d.type, dtype=np.int64)
    mask = np.isin(types_frame, keep_types)
    pos = np.asarray(d.pos_folded[mask], dtype=np.float32)
    if A == "HM":
        dip = np.asarray(d.dip[mask], dtype=np.float32)
    else:
        virt_mask = np.isin(types_frame,
                            [pds_virt_type, substrate_type] if substrate else [pds_virt_type])
        dip = np.asarray(d.dip[virt_mask], dtype=np.float32)
    return pos, dip

# 3 · VTKHDF — particle writers

Particles (+ dipoles, + optional bond lines) as a temporal VTKHDF `UnstructuredGrid`.
- `write_vtkhdf` — **time-sweep**: one h5 file, one frame per timestep.
- `write_vtkhdf_Hsweep` — **H-sweep**: one frame per h5 file at a fixed timestep.

## `write_vtkhdf` — Single-file time series (VTKHDF)

Writes a single **VTKHDF UnstructuredGrid** file readable by ParaView as a transient dataset.  
Bond topology written once; positions + dipoles streamed frame-by-frame (no full RAM load).

**Format notes:** requires `Type=UnstructuredGrid`, `Version=[2,0]`, and a `Steps` group.  
`Cells/Offsets` has `n_bonds+1` elements `[0, 2, 4, …, 2*n_bonds]` — VTK standard.

| Parameter | Description |
|---|---|
| `frames` | explicit list of integer timesteps |
| `A` | `"HM"` type-61 (all particles) · `"SM"` type-62 pos + type-666 dip |
| `particle_group` | H5 particle group (default `"Elastomer"`) |
| `overwrite` | if `False`, skip if output file already exists |

**VTKHDF File Structure**

total_active_bonds -> sum of active bonds for all frames
```
VTKHDF/
├── Cells/
│   ├── Connectivity                # (2 * total_active_bonds,)
│   ├── Offsets                     # (total_active_bonds + 1,)
│   └── Types                       # (total_active_bonds,)
├── NumberOfPoints                  # (n_frames,)
├── NumberOfCells                   # (n_frames,)
├── Points                          # (n_frames * n_points, 3)
├── PointData/
│   └── dipoles                     # (n_frames * n_points, 3)
└── Steps/
    ├── Values                      # (n_frames,)
    ├── PointOffsets                # (n_frames,)
    ├── CellOffsets                 # (n_frames,)
    ├── ConnectivityIdOffsets       # (n_frames,)
    └── FieldDataOffsets            # (n_frames,)
```

In [ ]:
def write_vtkhdf(h5_path, h5_out_path, frames,
                 particle_group="Elastomer", A="HM",
                 substrate=True, bonds=True,
                 overwrite=False):
    """
    Write a temporal VTKHDF UnstructuredGrid (fixed h5 file, one frame per timestep).

    - Always writes point cells (VTK_VERTEX) for particles.
    - When bonds=True, additionally writes bond cells (VTK_LINE).
    - Bonds crossing periodic boundaries in x/y are skipped.
    """

    if not overwrite and os.path.exists(h5_out_path):
        print(f"[skip] {h5_out_path} already exists")
        return 1

    if len(frames) == 0:
        raise ValueError("frames must contain at least one frame")

    os.makedirs(os.path.dirname(os.path.abspath(h5_out_path)), exist_ok=True)

    VTK_VERTEX = 1
    VTK_LINE   = 3

    pdp_real_type = 61
    pds_real_type = 62
    substrate_type = 98

    real_type = pdp_real_type if A == "HM" else pds_real_type

    with h5py.File(h5_path, "r") as h5_in:
        data  = H5DataSelector(h5_in, particle_group=particle_group)
        data0 = data.time(frames[0])

        assert set(frames).issubset(set(data._times_array)), f"{set(frames) - set(data._times_array)} missing in h5 data."

        box_l = np.asarray(h5_in["sys"].attrs["box_l"])
        bond_view_cutoff = 0.5 * box_l; bond_view_cutoff[2] = box_l[2]

        types0      = np.asarray(data0.type)
        keep_types  = [real_type, substrate_type] if substrate else [real_type]
        mask0       = np.isin(types0, keep_types)
        ids         = np.asarray(data0.id[mask0], dtype=np.int64)

        n_points = int(ids.size)
        n_frames = int(len(frames))

        # ── Bond topology (fixed to first frame; fast object-selection read) ─
        bonds_idx = np.empty((0, 2), dtype=np.int64)
        if bonds:
            id_to_index = {int(pid): i for i, pid in enumerate(ids)}
            bonds_idx = get_bond_index_pairs(data, A, id_to_index)

        # ── Pre-allocate per-step count arrays ───────────────────────────────
        num_points_per_step   = np.full(n_frames, n_points, dtype=np.int64)
        num_cells_per_step    = np.zeros(n_frames, dtype=np.int64)
        num_conn_ids_per_step = np.zeros(n_frames, dtype=np.int64)
        cell_offsets          = np.zeros((n_frames, 1), dtype=np.int64)
        conn_id_offsets       = np.zeros((n_frames, 1), dtype=np.int64)

        # ── Accumulation buffers ─────────────────────────────────────────────
        conn_all    = []   # flat connectivity
        offsets_all = []   # per-cell cumulative offsets (reset to 0 each step)
        cell_types  = []

        total_points = n_frames * n_points
        pts_chunk    = (max(n_points, 1), 3)

        with h5py.File(h5_out_path, "w") as h5_out:
            vtk = h5_out.create_group("VTKHDF")
            vtk.attrs["Type"] = np.bytes_("UnstructuredGrid")
            vtk.attrs["Version"] = np.array([2, 2], dtype=np.int64)

            vtk.create_dataset("NumberOfPoints", data=num_points_per_step)

            pts_ds = vtk.create_dataset(
                "Points", shape=(total_points, 3),
                dtype=np.float32, chunks=pts_chunk,
            )

            pd_grp = vtk.create_group("PointData")
            pd_grp.attrs["Vectors"] = np.bytes_("dipoles")
            dip_ds = pd_grp.create_dataset(
                "dipoles", shape=(total_points, 3),
                dtype=np.float32, chunks=pts_chunk,
            )

            steps = vtk.create_group("Steps")
            steps.attrs["NSteps"] = n_frames
            steps.create_dataset("Values", data=np.asarray(frames, dtype=np.float64))
            steps.create_dataset("PartOffsets", data=np.arange(n_frames, dtype=np.int64))
            steps.create_dataset("NumberOfParts", data=np.ones(n_frames, dtype=np.int64))
            steps.create_dataset("PointOffsets", data=np.arange(n_frames, dtype=np.int64) * n_points)

            pd_offsets = steps.create_group("PointDataOffsets")
            pd_offsets.create_dataset("dipoles", data=np.arange(n_frames, dtype=np.int64) * n_points)

            current_cell_offset = 0
            current_conn_offset = 0

            for ti, frame in enumerate(frames):
                d = data.time(frame)
                pos, dip = get_frame_particles(d, A, keep_types, substrate)

                if pos.shape[0] != n_points:
                    raise ValueError(f"frame {frame}: expected {n_points} pos, got {pos.shape[0]}")
                if dip.shape[0] != n_points:
                    raise ValueError(f"frame {frame}: expected {n_points} dip, got {dip.shape[0]}")

                s, e = ti * n_points, (ti + 1) * n_points
                pts_ds[s:e] = pos
                dip_ds[s:e] = dip

                # --- cell topology offsets for this step ---
                cell_offsets[ti, 0] = current_cell_offset
                conn_id_offsets[ti, 0] = current_conn_offset

                # ── Per-step cell data ────────────────────────────────────────
                # Offsets are cumulative within each step, starting from 0
                running_conn = 0  # cumulative connectivity count within this step

                # Fencepost: each step's Offsets section begins with 0
                offsets_all.append(0)

                # Vertex cells (one connectivity ID each)
                for p in range(n_points):
                    conn_all.append(p)
                    running_conn += 1
                    offsets_all.append(running_conn)   # 1, 2, 3, …, n_points
                    cell_types.append(VTK_VERTEX)

                n_step_cells = n_points
                n_step_conn_ids = n_points

                # Bond cells (two connectivity IDs each)
                n_active = 0
                if bonds and bonds_idx.size:
                    for i, j in bonds_idx:
                        vec = min_img_dist(pos[i], pos[j], box_dim=box_l)
                        # Skip bonds whose minimum-image displacement exceeds the per-axis cutoff (= PBC-wrapped bonds in x/y)
                        if np.all(np.abs(vec) < bond_view_cutoff):
                            conn_all.extend([i, j])
                            running_conn += 2
                            offsets_all.append(running_conn)  # n_points+2, +4,...
                            cell_types.append(VTK_LINE)
                            n_active += 1

                    n_step_cells    += n_active
                    n_step_conn_ids += 2 * n_active

                print(f" frame {frame}: {n_active} bonds", end="\r")

                num_cells_per_step[ti] = n_step_cells
                num_conn_ids_per_step[ti] = n_step_conn_ids
                current_cell_offset += n_step_cells
                current_conn_offset += n_step_conn_ids

            # ── Write cell datasets after all frames are processed ────────────
            vtk.create_dataset("NumberOfCells",             data=num_cells_per_step)
            vtk.create_dataset("NumberOfConnectivityIds",   data=num_conn_ids_per_step)

            # Connectivity
            vtk.create_dataset("Connectivity", data=np.asarray(conn_all,    dtype=np.int64))
            vtk.create_dataset("Offsets", data=np.asarray(offsets_all, dtype=np.int64))
            vtk.create_dataset("Types", data=np.asarray(cell_types,  dtype=np.uint8))

            # Cell offsets
            steps.create_dataset("CellOffsets", data=cell_offsets)
            steps.create_dataset("ConnectivityIdOffsets", data=conn_id_offsets)

    print(f"\nWritten {h5_out_path}")
    return 0

In [ ]:
def write_vtkhdf_Hsweep(h5_paths, h5_out_path, values, timestep=500,
                        particle_group="Elastomer", A="HM",
                        substrate=True, bonds=True, overwrite=False):
    """
    Like write_vtkhdf, but each 'frame' is a FIXED timestep read from a DIFFERENT h5 file —
    e.g. a field sweep: fixed t, H increasing, one file per H. `h5_paths` and `values` are
    parallel lists (values -> Steps/Values, the animation axis). Opens one file per frame.

    box_l / n_points / bond topology are taken from the first file (assumed constant across
    the sweep; a per-frame n_points check guards mismatches).
    """
    if not overwrite and os.path.exists(h5_out_path):
        print(f"[skip] {h5_out_path} already exists")
        return 1
    if len(h5_paths) == 0:
        raise ValueError("h5_paths must contain at least one path")
    if len(h5_paths) != len(values):
        raise ValueError("h5_paths and values must have the same length")

    os.makedirs(os.path.dirname(os.path.abspath(h5_out_path)), exist_ok=True)

    VTK_VERTEX = 1
    VTK_LINE   = 3
    pdp_real_type = 61
    pds_real_type = 62
    substrate_type = 98
    real_type = pdp_real_type if A == "HM" else pds_real_type

    n_frames   = int(len(h5_paths))
    keep_types = [real_type, substrate_type] if substrate else [real_type]

    # ── setup from the first file ──
    with h5py.File(h5_paths[0], "r") as h5_in:
        data = H5DataSelector(h5_in, particle_group=particle_group)
        assert timestep in set(data._times_array), f"timestep {timestep} missing in {h5_paths[0]}"
        box_l = np.asarray(h5_in["sys"].attrs["box_l"])
        bond_view_cutoff = 0.5 * box_l; bond_view_cutoff[2] = box_l[2]
        d0 = data.time(timestep)
        types0 = np.asarray(d0.type)
        mask0  = np.isin(types0, keep_types)
        ids    = np.asarray(d0.id[mask0], dtype=np.int64)
        n_points = int(ids.size)
        bonds_idx = np.empty((0, 2), dtype=np.int64)
        if bonds:
            id_to_index = {int(pid): i for i, pid in enumerate(ids)}
            bonds_idx = get_bond_index_pairs(data, A, id_to_index)

    total_points = n_frames * n_points
    pts_chunk    = (max(n_points, 1), 3)
    num_points_per_step   = np.full(n_frames, n_points, dtype=np.int64)
    num_cells_per_step    = np.zeros(n_frames, dtype=np.int64)
    num_conn_ids_per_step = np.zeros(n_frames, dtype=np.int64)
    cell_offsets          = np.zeros((n_frames, 1), dtype=np.int64)
    conn_id_offsets       = np.zeros((n_frames, 1), dtype=np.int64)
    conn_all = []; offsets_all = []; cell_types = []

    with h5py.File(h5_out_path, "w") as h5_out:
        vtk = h5_out.create_group("VTKHDF")
        vtk.attrs["Type"] = np.bytes_("UnstructuredGrid")
        vtk.attrs["Version"] = np.array([2, 2], dtype=np.int64)
        vtk.create_dataset("NumberOfPoints", data=num_points_per_step)
        pts_ds = vtk.create_dataset("Points", shape=(total_points, 3), dtype=np.float32, chunks=pts_chunk)
        pd_grp = vtk.create_group("PointData"); pd_grp.attrs["Vectors"] = np.bytes_("dipoles")
        dip_ds = pd_grp.create_dataset("dipoles", shape=(total_points, 3), dtype=np.float32, chunks=pts_chunk)

        steps = vtk.create_group("Steps")
        steps.attrs["NSteps"] = n_frames
        steps.create_dataset("Values", data=np.asarray(values, dtype=np.float64))
        steps.create_dataset("PartOffsets", data=np.arange(n_frames, dtype=np.int64))
        steps.create_dataset("NumberOfParts", data=np.ones(n_frames, dtype=np.int64))
        steps.create_dataset("PointOffsets", data=np.arange(n_frames, dtype=np.int64) * n_points)
        pd_offsets = steps.create_group("PointDataOffsets")
        pd_offsets.create_dataset("dipoles", data=np.arange(n_frames, dtype=np.int64) * n_points)

        current_cell_offset = 0
        current_conn_offset = 0
        for ti, (path, value) in enumerate(zip(h5_paths, values)):
            with h5py.File(path, "r") as h5_f:
                dsel = H5DataSelector(h5_f, particle_group=particle_group)
                assert timestep in set(dsel._times_array), f"timestep {timestep} missing in {path}"
                d = dsel.time(timestep)
                pos, dip = get_frame_particles(d, A, keep_types, substrate)

            if pos.shape[0] != n_points:
                raise ValueError(f"{path}: expected {n_points} pos, got {pos.shape[0]}")
            if dip.shape[0] != n_points:
                raise ValueError(f"{path}: expected {n_points} dip, got {dip.shape[0]}")

            s, e = ti * n_points, (ti + 1) * n_points
            pts_ds[s:e] = pos
            dip_ds[s:e] = dip

            cell_offsets[ti, 0] = current_cell_offset
            conn_id_offsets[ti, 0] = current_conn_offset
            running_conn = 0
            offsets_all.append(0)
            for p in range(n_points):
                conn_all.append(p)
                running_conn += 1
                offsets_all.append(running_conn)
                cell_types.append(VTK_VERTEX)
            n_step_cells = n_points
            n_step_conn_ids = n_points

            n_active = 0
            if bonds and bonds_idx.size:
                for i, j in bonds_idx:
                    vec = min_img_dist(pos[i], pos[j], box_dim=box_l)
                    if np.all(np.abs(vec) < bond_view_cutoff):
                        conn_all.extend([i, j])
                        running_conn += 2
                        offsets_all.append(running_conn)
                        cell_types.append(VTK_LINE)
                        n_active += 1
                n_step_cells    += n_active
                n_step_conn_ids += 2 * n_active

            print(f" {A} value={value}: {n_active} bonds", end="\r")
            num_cells_per_step[ti] = n_step_cells
            num_conn_ids_per_step[ti] = n_step_conn_ids
            current_cell_offset += n_step_cells
            current_conn_offset += n_step_conn_ids

        vtk.create_dataset("NumberOfCells",           data=num_cells_per_step)
        vtk.create_dataset("NumberOfConnectivityIds", data=num_conn_ids_per_step)
        vtk.create_dataset("Connectivity", data=np.asarray(conn_all,    dtype=np.int64))
        vtk.create_dataset("Offsets",      data=np.asarray(offsets_all, dtype=np.int64))
        vtk.create_dataset("Types",        data=np.asarray(cell_types,  dtype=np.uint8))
        steps.create_dataset("CellOffsets", data=cell_offsets)
        steps.create_dataset("ConnectivityIdOffsets", data=conn_id_offsets)

    print(f"\nWritten {h5_out_path}")
    return 0

# 4 · VTKHDF — surface writers

ITIM height-map surface (a `VTK_TRIANGLE` mesh) as a temporal VTKHDF. `bonds=True` adds the
bond-particle beads to the ITIM input so bond material lifts the surface. Needs module-globals
`SIZE` / `SIZE_BOND_PART` (set in section 7).
- `write_surface_vtkhdf` — **time-sweep**.
- `write_surface_vtkhdf_Hsweep` — **H-sweep**.

## `write_surface_vtkhdf` — Standalone ITIM surface file

Writes a single **VTKHDF UnstructuredGrid** holding the ITIM **height-map surface** for each
frame — a triangulated `VTK_TRIANGLE` mesh — in one file, **independent** of `write_vtkhdf`
(run them separately). It renders directly as a surface in ParaView.

Runs ITIM on the **real** elastomer particles (type 61/62; substrate excluded). With
**`bonds=True`**, the bond particles (beads laid every `0.25` along each bond, radius
`SIZE_BOND_PART/2`) are added to the ITIM input so bond material lifts the surface — the same
effect as `mae_analysis`'s with-bonds surface. Bond topology is read once via
`select_particles_by_object`. Uses the combined pyanal entry point `detect_surface_itim` and
module-globals `SIZE` / `SIZE_BOND_PART`; the surface floor `z0` is fixed inside. Cell topology
is constant, so it streams into fixed-size datasets → bounded RAM regardless of frame count.

| Parameter | Description |
|---|---|
| `frames` | explicit list of integer timesteps |
| `A` | `"HM"` real type 61 · `"SM"` real type 62 |
| `bonds` | include bond-particle beads in the ITIM input (lifts the surface) |
| `particle_group` | H5 particle group (default `"Elastomer"`) |
| `overwrite` | if `False`, skip if output file already exists |

> Requires module-globals `SIZE` and `SIZE_BOND_PART` (set in the run cell below).

In [ ]:
def write_surface_vtkhdf(h5_path, surf_out_path, frames,
                         particle_group="Elastomer", A="HM",
                         bonds=False, overwrite=False):
    """
    Write a standalone temporal VTKHDF UnstructuredGrid holding the ITIM
    height-map surface (a VTK_TRIANGLE mesh) for each frame — one file,
    independent of write_vtkhdf, that renders directly as a surface in ParaView.

    Runs ITIM on the real particles (type 61/62; substrate excluded). When
    bonds=True, the bond "particles" (beads laid every 0.25 along each bond,
    radius SIZE_BOND_PART/2) are added to the ITIM input so bond material lifts
    the surface — same effect as mae_analysis's with-bonds surface. Uses the
    module-globals SIZE and SIZE_BOND_PART for the probe geometry; floor z0 is
    fixed inside. Streams frame-by-frame into fixed-size datasets (bounded RAM).
    """

    if not overwrite and os.path.exists(surf_out_path):
        print(f"[skip] {surf_out_path} already exists")
        return 1

    if len(frames) == 0:
        raise ValueError("frames must contain at least one frame")

    os.makedirs(os.path.dirname(os.path.abspath(surf_out_path)), exist_ok=True)

    VTK_TRIANGLE = 5

    pdp_real_type = 61
    pds_real_type = 62
    real_type = pdp_real_type if A == "HM" else pds_real_type

    # ── ITIM probe geometry (mae_analysis convention, from module globals) ──
    radius             = SIZE / 2.0
    bond_radius        = SIZE_BOND_PART / 2.0
    bond_part_distance = 0.25
    ITIM_DIAMETER = SIZE * 0.2
    ball_R        = ITIM_DIAMETER / 2.0
    grid_spacing  = ITIM_DIAMETER / 4.0
    itim_z0       = 1.0

    with h5py.File(h5_path, "r") as h5_in:
        data = H5DataSelector(h5_in, particle_group=particle_group)

        assert set(frames).issubset(set(data._times_array)), f"{set(frames) - set(data._times_array)} missing in h5 data."

        box_l = np.asarray(h5_in["sys"].attrs["box_l"])

        max_r = max(radius, bond_radius) if bonds else radius
        assert ball_R > grid_spacing, "ITIM ball radius (SIZE*0.1) must exceed grid spacing (SIZE*0.05)"
        assert 2.0 * (ball_R + max_r) < min(box_l[0], box_l[1]), \
            "ITIM cutoff (R + particle radius) too large for box (needs 2*(R+radius) < min(box_l_x, box_l_y))"

        nx = math.ceil(box_l[0] / grid_spacing)
        ny = math.ceil(box_l[1] / grid_spacing)
        n_mesh_pts = nx * ny
        # grid_bottom must be a full (nx, ny) array: the public ITIM API round-trips
        # it through np.save/np.load, which turns a Python scalar into a 0-d array
        # and breaks pyanal's isinstance(grid_bottom, (int, float)) branch.
        itim_grid_bottom = np.full((nx, ny), itim_z0, dtype=np.float32)

        data0      = data.time(frames[0])
        types0     = np.asarray(data0.type, dtype=np.int64)
        real_mask0 = types0 == real_type

        # ── Bond topology (static; read ONCE via the fast object-selection helper) ──
        bond_pairs = np.empty((0, 2), dtype=np.int64)
        if bonds:
            real_ids   = np.asarray(data0.id[real_mask0], dtype=np.int64)
            id_to_ridx = {int(pid): i for i, pid in enumerate(real_ids)}
            bond_pairs = get_bond_index_pairs(data, A, id_to_ridx)

        # ── Triangle mesh topology of the (nx, ny) lattice — constant across frames.
        #    p(i, j) = i*ny + j indexes X/Y/Z.ravel() (indexing='ij'); 2 triangles/cell. ──
        ii, jj = np.meshgrid(np.arange(nx - 1), np.arange(ny - 1), indexing="ij")
        p00 = (ii * ny + jj).ravel()
        p01 = (ii * ny + (jj + 1)).ravel()
        p10 = ((ii + 1) * ny + jj).ravel()
        p11 = ((ii + 1) * ny + (jj + 1)).ravel()
        tri = np.empty((p00.size * 2, 3), dtype=np.int64)
        tri[0::2, 0] = p00; tri[0::2, 1] = p10; tri[0::2, 2] = p11
        tri[1::2, 0] = p00; tri[1::2, 1] = p11; tri[1::2, 2] = p01
        tri_conn    = tri.ravel()                                  # (3*n_tri,)
        n_tri       = tri_conn.size // 3
        tri_offsets = np.arange(0, 3 * n_tri + 1, 3, dtype=np.int64)   # (n_tri+1,)
        tri_types   = np.full(n_tri, VTK_TRIANGLE, dtype=np.uint8)

        n_frames = int(len(frames))

        with h5py.File(surf_out_path, "w") as h5_out:
            vtk = h5_out.create_group("VTKHDF")
            vtk.attrs["Type"] = np.bytes_("UnstructuredGrid")
            vtk.attrs["Version"] = np.array([2, 2], dtype=np.int64)

            # Constant topology → fixed-size datasets, written one frame's slice
            # at a time (never np.tile the whole trajectory into RAM).
            vtk.create_dataset("NumberOfPoints",          data=np.full(n_frames, n_mesh_pts, dtype=np.int64))
            vtk.create_dataset("NumberOfCells",           data=np.full(n_frames, n_tri, dtype=np.int64))
            vtk.create_dataset("NumberOfConnectivityIds", data=np.full(n_frames, 3 * n_tri, dtype=np.int64))

            pts_ds  = vtk.create_dataset("Points", shape=(n_frames * n_mesh_pts, 3),
                                         dtype=np.float32, chunks=(max(n_mesh_pts, 1), 3))
            conn_ds = vtk.create_dataset("Connectivity", shape=(n_frames * 3 * n_tri,),
                                         dtype=np.int64, chunks=(max(3 * n_tri, 1),))
            off_ds  = vtk.create_dataset("Offsets", shape=(n_frames * (n_tri + 1),),
                                         dtype=np.int64, chunks=(max(n_tri + 1, 1),))
            typ_ds  = vtk.create_dataset("Types", shape=(n_frames * n_tri,),
                                         dtype=np.uint8, chunks=(max(n_tri, 1),))

            steps = vtk.create_group("Steps")
            steps.attrs["NSteps"] = n_frames
            steps.create_dataset("Values", data=np.asarray(frames, dtype=np.float64))
            steps.create_dataset("PartOffsets", data=np.arange(n_frames, dtype=np.int64))
            steps.create_dataset("NumberOfParts", data=np.ones(n_frames, dtype=np.int64))
            steps.create_dataset("PointOffsets", data=np.arange(n_frames, dtype=np.int64) * n_mesh_pts)
            steps.create_dataset("CellOffsets", data=(np.arange(n_frames, dtype=np.int64) * n_tri).reshape(-1, 1))
            steps.create_dataset("ConnectivityIdOffsets", data=(np.arange(n_frames, dtype=np.int64) * 3 * n_tri).reshape(-1, 1))

            for ti, frame in enumerate(frames):
                d = data.time(frame)
                types_frame = np.asarray(d.type, dtype=np.int64)
                pos_real = np.asarray(d.pos_folded[types_frame == real_type], dtype=np.float32)

                # ── ITIM input: real particles (+ bond beads when requested) ──
                if bonds and bond_pairs.size:
                    beads = calculate_bond_particles(pos_real, bond_pairs, box_l, bond_part_distance)
                    pos_itim   = np.vstack([pos_real, beads])
                    radii_itim = np.concatenate([
                        np.full(pos_real.shape[0], radius,      dtype=np.float32),
                        np.full(beads.shape[0],    bond_radius, dtype=np.float32)])
                else:
                    pos_itim   = pos_real
                    radii_itim = np.full(pos_real.shape[0], radius, dtype=np.float32)

                X, Y, Z, _ = detect_surface_itim(pos_itim, radii_itim, R=ball_R,
                                                 box_l=box_l, grid_bottom=itim_grid_bottom)
                if ti == 0:
                    assert X.shape == (nx, ny), f"ITIM grid shape mismatch: expected {(nx, ny)}, got {X.shape}"

                # ── Write this frame's slice (height-map mesh only) ──
                pts_ds[ti * n_mesh_pts:(ti + 1) * n_mesh_pts] = \
                    np.column_stack([X.ravel(), Y.ravel(), Z.ravel()]).astype(np.float32)
                conn_ds[ti * 3 * n_tri:(ti + 1) * 3 * n_tri] = tri_conn
                off_ds[ti * (n_tri + 1):(ti + 1) * (n_tri + 1)] = tri_offsets
                typ_ds[ti * n_tri:(ti + 1) * n_tri] = tri_types

                print(f" frame {frame}: surface{' (+bonds)' if (bonds and bond_pairs.size) else ''}", end="\r")

    print(f"\nWritten {surf_out_path}")
    return 0

In [ ]:
def write_surface_vtkhdf_Hsweep(h5_paths, surf_out_path, values, timestep=500,
                                particle_group="Elastomer", A="HM",
                                bonds=False, overwrite=False):
    """
    Like write_surface_vtkhdf, but each frame is a FIXED timestep read from a DIFFERENT h5
    file (e.g. an H sweep at fixed t). `h5_paths` and `values` are parallel lists
    (values -> Steps/Values). Mesh-only (VTK_TRIANGLE), fixed-size streaming. Bond beads
    (bonds=True) use the static topology / box from the first file. Uses module-globals
    SIZE / SIZE_BOND_PART.
    """
    if not overwrite and os.path.exists(surf_out_path):
        print(f"[skip] {surf_out_path} already exists")
        return 1
    if len(h5_paths) == 0:
        raise ValueError("h5_paths must contain at least one path")
    if len(h5_paths) != len(values):
        raise ValueError("h5_paths and values must have the same length")

    os.makedirs(os.path.dirname(os.path.abspath(surf_out_path)), exist_ok=True)

    VTK_TRIANGLE = 5
    pdp_real_type = 61
    pds_real_type = 62
    real_type = pdp_real_type if A == "HM" else pds_real_type

    radius             = SIZE / 2.0
    bond_radius        = SIZE_BOND_PART / 2.0
    bond_part_distance = 0.25
    ITIM_DIAMETER = SIZE * 0.2
    ball_R        = ITIM_DIAMETER / 2.0
    grid_spacing  = ITIM_DIAMETER / 4.0
    itim_z0       = 1.0
    n_frames = int(len(h5_paths))

    # ── setup from the first file ──
    with h5py.File(h5_paths[0], "r") as h5_in:
        data = H5DataSelector(h5_in, particle_group=particle_group)
        assert timestep in set(data._times_array), f"timestep {timestep} missing in {h5_paths[0]}"
        box_l = np.asarray(h5_in["sys"].attrs["box_l"])
        max_r = max(radius, bond_radius) if bonds else radius
        assert ball_R > grid_spacing, "ITIM ball radius (SIZE*0.1) must exceed grid spacing (SIZE*0.05)"
        assert 2.0 * (ball_R + max_r) < min(box_l[0], box_l[1]), \
            "ITIM cutoff (R + particle radius) too large for box"
        nx = math.ceil(box_l[0] / grid_spacing)
        ny = math.ceil(box_l[1] / grid_spacing)
        n_mesh_pts = nx * ny
        itim_grid_bottom = np.full((nx, ny), itim_z0, dtype=np.float32)
        bond_pairs = np.empty((0, 2), dtype=np.int64)
        if bonds:
            d0 = data.time(timestep)
            types0 = np.asarray(d0.type, dtype=np.int64)
            real_ids = np.asarray(d0.id[types0 == real_type], dtype=np.int64)
            id_to_ridx = {int(pid): i for i, pid in enumerate(real_ids)}
            bond_pairs = get_bond_index_pairs(data, A, id_to_ridx)

    # ── triangle mesh topology (constant across frames) ──
    ii, jj = np.meshgrid(np.arange(nx - 1), np.arange(ny - 1), indexing="ij")
    p00 = (ii * ny + jj).ravel()
    p01 = (ii * ny + (jj + 1)).ravel()
    p10 = ((ii + 1) * ny + jj).ravel()
    p11 = ((ii + 1) * ny + (jj + 1)).ravel()
    tri = np.empty((p00.size * 2, 3), dtype=np.int64)
    tri[0::2, 0] = p00; tri[0::2, 1] = p10; tri[0::2, 2] = p11
    tri[1::2, 0] = p00; tri[1::2, 1] = p11; tri[1::2, 2] = p01
    tri_conn    = tri.ravel()
    n_tri       = tri_conn.size // 3
    tri_offsets = np.arange(0, 3 * n_tri + 1, 3, dtype=np.int64)
    tri_types   = np.full(n_tri, VTK_TRIANGLE, dtype=np.uint8)

    with h5py.File(surf_out_path, "w") as h5_out:
        vtk = h5_out.create_group("VTKHDF")
        vtk.attrs["Type"] = np.bytes_("UnstructuredGrid")
        vtk.attrs["Version"] = np.array([2, 2], dtype=np.int64)
        vtk.create_dataset("NumberOfPoints",          data=np.full(n_frames, n_mesh_pts, dtype=np.int64))
        vtk.create_dataset("NumberOfCells",           data=np.full(n_frames, n_tri, dtype=np.int64))
        vtk.create_dataset("NumberOfConnectivityIds", data=np.full(n_frames, 3 * n_tri, dtype=np.int64))
        pts_ds  = vtk.create_dataset("Points", shape=(n_frames * n_mesh_pts, 3),
                                     dtype=np.float32, chunks=(max(n_mesh_pts, 1), 3))
        conn_ds = vtk.create_dataset("Connectivity", shape=(n_frames * 3 * n_tri,),
                                     dtype=np.int64, chunks=(max(3 * n_tri, 1),))
        off_ds  = vtk.create_dataset("Offsets", shape=(n_frames * (n_tri + 1),),
                                     dtype=np.int64, chunks=(max(n_tri + 1, 1),))
        typ_ds  = vtk.create_dataset("Types", shape=(n_frames * n_tri,),
                                     dtype=np.uint8, chunks=(max(n_tri, 1),))
        steps = vtk.create_group("Steps")
        steps.attrs["NSteps"] = n_frames
        steps.create_dataset("Values", data=np.asarray(values, dtype=np.float64))
        steps.create_dataset("PartOffsets", data=np.arange(n_frames, dtype=np.int64))
        steps.create_dataset("NumberOfParts", data=np.ones(n_frames, dtype=np.int64))
        steps.create_dataset("PointOffsets", data=np.arange(n_frames, dtype=np.int64) * n_mesh_pts)
        steps.create_dataset("CellOffsets", data=(np.arange(n_frames, dtype=np.int64) * n_tri).reshape(-1, 1))
        steps.create_dataset("ConnectivityIdOffsets", data=(np.arange(n_frames, dtype=np.int64) * 3 * n_tri).reshape(-1, 1))

        for ti, (path, value) in enumerate(zip(h5_paths, values)):
            with h5py.File(path, "r") as h5_f:
                dsel = H5DataSelector(h5_f, particle_group=particle_group)
                assert timestep in set(dsel._times_array), f"timestep {timestep} missing in {path}"
                d = dsel.time(timestep)
                types_frame = np.asarray(d.type, dtype=np.int64)
                pos_real = np.asarray(d.pos_folded[types_frame == real_type], dtype=np.float32)

            if bonds and bond_pairs.size:
                beads = calculate_bond_particles(pos_real, bond_pairs, box_l, bond_part_distance)
                pos_itim   = np.vstack([pos_real, beads])
                radii_itim = np.concatenate([
                    np.full(pos_real.shape[0], radius,      dtype=np.float32),
                    np.full(beads.shape[0],    bond_radius, dtype=np.float32)])
            else:
                pos_itim   = pos_real
                radii_itim = np.full(pos_real.shape[0], radius, dtype=np.float32)

            X, Y, Z, _ = detect_surface_itim(pos_itim, radii_itim, R=ball_R,
                                             box_l=box_l, grid_bottom=itim_grid_bottom)
            if ti == 0:
                assert X.shape == (nx, ny), f"ITIM grid shape mismatch: expected {(nx, ny)}, got {X.shape}"

            pts_ds[ti * n_mesh_pts:(ti + 1) * n_mesh_pts] = \
                np.column_stack([X.ravel(), Y.ravel(), Z.ravel()]).astype(np.float32)
            conn_ds[ti * 3 * n_tri:(ti + 1) * 3 * n_tri] = tri_conn
            off_ds[ti * (n_tri + 1):(ti + 1) * (n_tri + 1)] = tri_offsets
            typ_ds[ti * n_tri:(ti + 1) * n_tri] = tri_types

            print(f" {A} value={value}: surface{' (+bonds)' if (bonds and bond_pairs.size) else ''}", end="\r")

    print(f"\nWritten {surf_out_path}")
    return 0

# 5 · Other exporters

Alternative output formats (kept for convenience):
- `export_vtp_pvd` — per-frame `.vtp` + a `.pvd` collection.
- `write_vtk_frame_modern` / `write_vtk_box` — legacy single-frame ASCII `.vtk`.

## `export_vtp_pvd` — Per-frame VTP + PVD collection

Writes one `.vtp` per timestep plus a `collection.pvd` index file for ParaView.

| Parameter | Description |
|---|---|
| `frames` | explicit list of integer timesteps |
| `A` | `"HM"` type-61 (all particles) · `"SM"` type-62 pos + type-666 dip |
| `particle_group` | H5 particle group (default `"Elastomer"`) |
| `save_bonds` | embed bond lines in each VTP frame (computed once from frame 0) |
| `save_substrate` | write `substrate.vtp` from type-98 particles |
| `overwrite` | if `False`, skip frames that already exist on disk |

In [ ]:
def write_vtp_frame(positions, dipoles, bonds_idx, path):
    n_points = len(positions)
    n_bonds  = len(bonds_idx) if bonds_idx is not None else 0

    with open(path, "w") as f:
        f.write('<?xml version="1.0"?>\n')
        f.write('<VTKFile type="PolyData" version="0.1" byte_order="LittleEndian">\n')
        f.write('  <PolyData>\n')
        f.write(f'    <Piece NumberOfPoints="{n_points}" NumberOfVerts="0" '
                f'NumberOfLines="{n_bonds}" NumberOfStrips="0" NumberOfPolys="0">\n')

        f.write('      <Points>\n')
        f.write('        <DataArray type="Float32" NumberOfComponents="3" format="ascii">\n')
        np.savetxt(f, positions, fmt="%.6f")
        f.write('        </DataArray>\n      </Points>\n')

        if n_bonds > 0:
            f.write('      <Lines>\n')
            f.write('        <DataArray type="Int32" Name="connectivity" format="ascii">\n')
            f.write(' '.join(f'{i} {j}' for i, j in bonds_idx) + '\n')
            f.write('        </DataArray>\n')
            f.write('        <DataArray type="Int32" Name="offsets" format="ascii">\n')
            f.write(' '.join(str(2 * k) for k in range(1, n_bonds + 1)) + '\n')
            f.write('        </DataArray>\n      </Lines>\n')

        f.write('      <PointData Vectors="dipoles">\n')
        f.write('        <DataArray type="Float32" NumberOfComponents="3" Name="dipoles" format="ascii">\n')
        np.savetxt(f, dipoles, fmt="%.6f")
        f.write('        </DataArray>\n      </PointData>\n')

        f.write('    </Piece>\n  </PolyData>\n</VTKFile>\n')


def write_substrate(positions, path):
    n_points = len(positions)
    with open(path, "w") as f:
        f.write('<?xml version="1.0"?>\n')
        f.write('<VTKFile type="PolyData" version="0.1" byte_order="LittleEndian">\n')
        f.write('  <PolyData>\n')
        f.write(f'    <Piece NumberOfPoints="{n_points}" NumberOfVerts="0" '
                'NumberOfLines="0" NumberOfStrips="0" NumberOfPolys="0">\n')
        f.write('      <Points>\n')
        f.write('        <DataArray type="Float32" NumberOfComponents="3" format="ascii">\n')
        np.savetxt(f, positions, fmt="%.6f")
        f.write('        </DataArray>\n      </Points>\n')
        f.write('    </Piece>\n  </PolyData>\n</VTKFile>\n')


def write_pvd(pvd_path, vtp_files, time_values):
    root       = etree.Element("VTKFile", type="Collection",
                               version="0.1", byte_order="LittleEndian")
    collection = etree.SubElement(root, "Collection")
    for fname, t in zip(vtp_files, time_values):
        etree.SubElement(collection, "DataSet",
                         timestep=str(t), group="", part="0",
                         file=os.path.basename(fname))
    etree.ElementTree(root).write(pvd_path, pretty_print=True,
                                  xml_declaration=True, encoding="UTF-8")


def export_vtp_pvd(h5_path, vtk_dir_path, frames,
                   A=None, particle_group="Elastomer",
                   save_bonds=False, save_substrate=True,
                   overwrite=False):
    os.makedirs(vtk_dir_path, exist_ok=True)

    with h5py.File(h5_path, "r") as h5_file:
        data = H5DataSelector(h5_file, particle_group=particle_group)

        missing = set(frames) - set(data.times_array.astype(int))
        assert not missing, f"Frames not in h5: {missing}"

        data0    = data.time(frames[0])
        types0   = data0.type
        mask_sub = types0 == 98

        if A == "HM":
            ids = data0.id
        else:
            ids = data0.id[types0 == 62]

        # Bond topology (fast object-selection read; shared helper)
        bonds_idx = None
        if save_bonds:
            id_to_index = {int(pid): i for i, pid in enumerate(np.asarray(ids))}
            bonds_idx = get_bond_index_pairs(data, A, id_to_index)

        if save_substrate:
            sub_path = os.path.join(vtk_dir_path, "substrate.vtp")
            if overwrite or not os.path.exists(sub_path):
                write_substrate(data0.pos_folded[mask_sub], sub_path)

        vtp_files = []
        for frame in frames:
            vtp_path = os.path.join(vtk_dir_path, f"t_{frame}.vtp")
            vtp_files.append(vtp_path)

            if not overwrite and os.path.exists(vtp_path):
                print(f"  [skip] frame {frame}")
                continue

            d = data.time(frame)
            if A == "HM":
                positions = d.pos_folded
                dipoles   = d.dip
            else:
                positions = d.pos_folded[d.type == 62]
                dipoles   = d.dip[d.type == 666]

            write_vtp_frame(positions, dipoles, bonds_idx, vtp_path)
            print(f"  frame {frame}", end="\r")

        pvd_path = os.path.join(vtk_dir_path, "collection.pvd")
        write_pvd(pvd_path, vtp_files, frames)
        print(f"\nWritten {pvd_path}")

## `write_vtk_frame_modern` + `write_vtk_box` — Legacy ASCII VTK

Single-frame `.vtk` files (VTK 2.0 ASCII format). **No bond support** — too slow for large systems.  
Loop manually for time series. Use for quick inspection or tools that don't support VTP/VTKHDF.

| Parameter | Description |
|---|---|
| `particle_group` | H5 group (default `"Elastomer"`) |
| `frame` | integer timestep index |
| `layer_height` | override box Z for `write_vtk_box` |

In [ ]:
def write_vtk_frame_modern(data_file, path_target="path", frame=0,
                           particle_group="Elastomer"):
    data = H5DataSelector(data_file, particle_group=particle_group)
    d    = data.time(frame)

    d = data.time(frame)
    if A == "HM":
        positions = d.pos_folded
        dipoles   = d.dip
    else:
        positions = d.pos_folded[(d.type == 62) | (d.type == 98)]
        dipoles   = d.dip[(d.type == 666) | (d.type == 98)]

    assert len(positions) == len(dipoles), \
        f"Real/virtual count mismatch: {len(positions)} vs {len(dipoles)}"

    with open(path_target, "w") as vtk:
        vtk.write("# vtk DataFile Version 2.0\nparticles\nASCII\n"
                  "DATASET UNSTRUCTURED_GRID\n")
        vtk.write(f"POINTS {len(positions)} floats\n")
        np.savetxt(vtk, positions, fmt="%.6f")
        vtk.write(f"POINT_DATA {len(positions)}\n"
                  "VECTORS dipoles float\n")
        np.savetxt(vtk, dipoles, fmt="%.6f")

    return 0


def write_vtk_box(data_file, path_target="path_box", layer_height=None):
    box_l = data_file["sys"].attrs["box_l"].copy()
    if layer_height is not None:
        box_l[2] = layer_height

    corners = np.array([
        [0,        0,        0        ],
        [box_l[0], 0,        0        ],
        [box_l[0], box_l[1], 0        ],
        [0,        box_l[1], 0        ],
        [0,        0,        box_l[2] ],
        [box_l[0], 0,        box_l[2] ],
        [box_l[0], box_l[1], box_l[2] ],
        [0,        box_l[1], box_l[2] ],
    ])
    edges = [
        [0, 1], [1, 2], [2, 3], [3, 0],
        [4, 5], [5, 6], [6, 7], [7, 4],
        [0, 4], [1, 5], [2, 6], [3, 7],
    ]
    with open(path_target, "w") as vtk:
        vtk.write("# vtk DataFile Version 2.0\nbox\nASCII\n"
                  "DATASET UNSTRUCTURED_GRID\n")
        vtk.write(f"POINTS {len(corners)} float\n")
        np.savetxt(vtk, corners, fmt="%.6f")
        vtk.write(f"CELLS {len(edges)} {len(edges) * 3}\n")
        for e in edges:
            vtk.write(f"2 {e[0]} {e[1]}\n")
        vtk.write(f"CELL_TYPES {len(edges)}\n")
        vtk.write("3\n" * len(edges))

# 6 · H5 exploration

Raw HDF5 inspection — keys, attributes, dataset shapes, and `H5DataSelector` API examples.
(Scratch / reference; not needed to produce videos.)

In [ ]:
import h5py
from pressomancy.analysis import H5DataSelector
from pressomancy.simulation import Elastomer, PointDipolePermanent, PointDipoleSuperpara
import numpy as np

A = "SM"

h5_path, _ = get_sim_paths(A=A, H=15, seed=1, dens=0.30, n_full_box=24000, height=10)
with h5py.File(h5_path, "r") as h5_file:
    # print all top-level groups
    print("Top-level keys:", list(h5_file.keys()))
    # e.g. -> ['particles', 'connectivity']

    # Explore inside sys
    print("\nSys:", list(h5_file["sys"].attrs))
    print({attr: h5_file["sys"].attrs[attr] for attr in list(h5_file["sys"].attrs)})

    # Explore inside connectivity
    print("\nConnectivity groups:", list(h5_file["connectivity"].keys()))
    # e.g. -> ['Elastomer', 'PointDipolePermanent']

    # Explore datasets inside one of them
    print("\nDatasets in connectivity/Elastomer:")
    print(list(h5_file["connectivity/Elastomer"].keys()))

    print("Datasets in connectivity/Elastomer/Elastomer_to_PointDipolePermanent:")
    print(list(h5_file[f"connectivity/Elastomer/Elastomer_to_{"PointDipolePermanent" if A=="HM" else "PointDipoleSuperpara"}"]))

    print("Datasets in connectivity/Elastomer/ParticleHandle_to_Elastomer:")
    print(list(h5_file["connectivity/Elastomer/ParticleHandle_to_Elastomer"]))

    print(f"Datasets in connectivity/Elastomer/ParticleHandle_to_{"PointDipolePermanent" if A=="HM" else "PointDipoleSuperpara"}:")
    print(list(h5_file[f"connectivity/Elastomer/ParticleHandle_to_{"PointDipolePermanent" if A=="HM" else "PointDipoleSuperpara"}"]))

    # Explore inside particles
    print("\n particles groups:", list(h5_file["particles"].keys()))
    print("\nDatasets in particles/Elastomer:")
    print(list(h5_file["particles/Elastomer"].keys()))
    print(list(h5_file["particles/Elastomer/bonds"].keys()))
    print(len(list(h5_file["particles/Elastomer/id/step"])))
    print("h5_file[\"particles/Elastomer/id/step\"]")
    print(np.asarray(h5_file["particles/Elastomer/id/step"]))
    print(len(list(h5_file["particles/Elastomer/id/time"])))
    print("h5_file[\"particles/Elastomer/id/time\"]")
    print(np.asarray(h5_file["particles/Elastomer/id/time"]))
    print(h5_file["particles/Elastomer/bonds/value"])
    print("h5_file[\"particles/Elastomer/bonds/step\"]")
    print(np.asarray(h5_file["particles/Elastomer/bonds/step"]))
    print(h5_file["particles/Elastomer/bonds/value"].shape)


    elastomer_h5 = H5DataSelector(h5_file=h5_file, particle_group="Elastomer")

    M_part = elastomer_h5.select_particles_by_object("PointDipolePermanent")
    print(M_part.timestep[0].id.shape)

    time=500
    if time == None:
        step = -1
    else:
        times = h5_file["particles/Elastomer/id/step"][:]
        step = np.nonzero(times == time)[0][0]

    types = h5_file["particles/Elastomer/type/value"][step,:,0]
    vip_type_mask = np.isin(types, (61, 3, 62, 4))
    ids = h5_file["particles/Elastomer/id/value"][step,:,0][vip_type_mask]
    poss = h5_file["particles/Elastomer/pos/value"][step,:,:][vip_type_mask]
    dips = h5_file["particles/Elastomer/dip/value"][step,:,:][vip_type_mask]
    # bonds require a bit more work for older pressomancy
    ds = h5_file["particles/Elastomer/bonds/value"]
    bond_full=[]
    for i in range(ds.shape[1]):
        try:
            assert len(ds[0, i]) > 0
            bond_full.append(ds[0, i])
        except:
            bond_full.append([])
    bonds = np.asarray([bond_full], dtype=object)[0,:][vip_type_mask]
    print("bondshape", bonds.shape)
    print(poss.shape, ids.shape)

    # M_part = elastomer_h5.select_particles_by_predicate(predicate=lambda ds: np.asarray(ds.timestep[0].get_property("type")) == 3)
    # print(M_part.timestep[:].particles[:])

    # print(h5_file["particles/Elastomer/id/value"].shape)
    # print(len(M_part.get_property('id')))

    # step_part_dip = np.asarray(M_part.get_property("dip"))
    # step_part_pos = np.asarray(M_part.get_property("pos"))

    # M_part_at_step = M_part.timestep[0].particles[:5]
    # print(M_part_at_step.pos.shape)
    # print(M_part_at_step.pos)
    # print(M_part_at_step.pos[:,:2])
    # M_part_at_step = M_part.timestep[0]
    # for part in M_part_at_step.particles:
    # #     print(part.id)
    # #     print(part.pos)
        # print(int(part.id), part.bonds)

    # test_get_bonds = M_part.timestep[:2].particles[:2].bonds
    # print(test_get_bonds)
    # print(test_get_bonds[0,:])

    # print([(part.id, part.pos) for part in M_part.timestep[100].particles[:3].particles])
    # print("\n")
    # print([(part.id, part.pos) for part in M_part.timestep[101].particles[:3].particles])

# 7 · Execution

Run whichever cells you need from here (all function defs live above).

The **first cell below is the main deliverable**: the MAE-vs-field video (t=500, H: 0 → 30) for
both HM and SM — particle `.vtkhdf` + surface `.vtkhdf` per model. The remaining cells are the
original time-sweep examples.

> Sets the module-globals `SIZE` / `SIZE_BOND_PART` used by the surface writers.

In [ ]:
# MAE-vs-field video — fixed t=500, H rises 0 → 30, for HM and SM ──
# Each frame is a different h5 file (one per H); in ParaView the time slider becomes H.
# Produces, per model: <A>_Hsweep.vtkhdf (particles) and <A>_Hsweep_surface.vtkhdf (surface).
SIZE = 1.
SIZE_BOND_PART = 0.9

H_list  = [0, 0.2, 0.4, 0.6, 0.8, 1, 1.5, 2, 3, 4, 6, 10, 15, 20, 25, 30]
t_fixed = 500
out_dir = os.path.join(DATA_DIR, "vtk_cluster/Hsweep")
os.makedirs(out_dir, exist_ok=True)

for A in ("HM", "SM"):
    h5_paths = [
        get_sim_paths(A=A, H=H, seed=1, dens=0.30, K="hard",
                      n_full_box=24000, height=10,
                      h5_root=os.path.join(DATA_DIR, "MAE-BoS/cluster/sim_data"))[0]
        for H in H_list
    ]
    # particle video (positions + dipoles + bonds)
    write_vtkhdf_Hsweep(h5_paths, os.path.join(out_dir, f"{A}_Hsweep.vtkhdf"),
                        H_list, timestep=t_fixed, A=A, bonds=True, overwrite=True)
    # surface video (ITIM height-map mesh, bond beads included)
    write_surface_vtkhdf_Hsweep(h5_paths, os.path.join(out_dir, f"{A}_Hsweep_surface.vtkhdf"),
                                H_list, timestep=t_fixed, A=A, bonds=True, overwrite=True)

### Time-sweep examples (fixed H, varying t)

Original per-writer example calls, collected here. Each sets its own paths/params — edit and run
whichever you need.

In [ ]:
# [write_vtkhdf] time-sweep particles (fixed H, varying t)
A="SM"

SIZE = 1.

h5_path, vtk_dir = get_sim_paths(
    A=A, H=6, seed=1,
    dens=0.20, K="soft",
    n_full_box=24000, height=10,
    h5_root=os.path.join(DATA_DIR, "MAE-BoS/cluster/sim_data"),
    ext=".h5"
)
vtk_out = os.path.join(vtk_dir, "trajectory.vtkhdf")
os.makedirs(vtk_dir, exist_ok=True)

frames = list(range(0, 500 + 1, 1))
write_vtkhdf(h5_path, vtk_out, frames, A=A, overwrite=True, bonds=True)

if False:
    import h5py

    def dump(name, obj):
        pad = "  " * name.count("/")
        if isinstance(obj, h5py.Dataset):
            val = obj[()] if obj.size <= 12 else f"shape={obj.shape}"
            print(f"{pad}{name}: {val}  [{obj.dtype}]")
        else:
            print(f"{pad}{name}/  attrs={dict(obj.attrs)}")

    with h5py.File(vtk_out, "r") as f:
        print(f"root attrs: {dict(f['VTKHDF'].attrs)}")
        f["VTKHDF"].visititems(dump)

In [ ]:
# [write_vtkhdf] time-sweep particles for an add_dipoles run
H=1e-5

h5_path = os.path.join(DATA_DIR, f"add_dipoles_run/h5/hcp/n1200-H{H}.h5")
vtk_dir = os.path.join(DATA_DIR, "add_dipoles_run/vtk_hoje")
vtk_out = os.path.join(vtk_dir, f"H{H}.vtkhdf")
os.makedirs(vtk_dir, exist_ok=True)

frames = list(range(0, 30 + 1, 1))
write_vtkhdf(h5_path, vtk_out, frames, A="SM", overwrite=True, particle_group="PointDipoleSuperpara", bonds=False)

In [ ]:
# [write_surface_vtkhdf] time-sweep ITIM surface (fixed H, varying t)
A = "HM"
SIZE = 1.
SIZE_BOND_PART = 0.9

h5_path, vtk_dir = get_sim_paths(
    A=A, H=0, seed=1,
    dens=0.30, K="soft",
    n_full_box=24000, height=10,
    h5_root=os.path.join(DATA_DIR, "MAE-BoS/cluster/sim_data"),
    ext=".h5"
)
surf_out = os.path.join(vtk_dir, "surface.vtkhdf")
os.makedirs(vtk_dir, exist_ok=True)

frames = list(range(0, 500 + 1, 1))
# bonds=True adds the bond-particle beads to the ITIM input (lifts the surface)
write_surface_vtkhdf(h5_path, surf_out, frames, A=A, bonds=True, overwrite=True)

In [ ]:
# [export_vtp_pvd] time-sweep VTP+PVD collection
A, H, seed = "SM", 15.0, 1

h5_path, vtk_dir = get_sim_paths(
    A=A, H=H, seed=seed,
    dens=0.30, n_full_box=24000, height=10
)

frames = list(range(0, 500 + 1, 1))
export_vtp_pvd(h5_path, vtk_dir, frames, A=A, save_bonds=True, overwrite=False)

In [ ]:
# [legacy ASCII VTK] per-frame .vtk for an add_dipoles run
lattice, n, H = "hcp", 12000, 0.8

h5_path = os.path.join(DATA_DIR, f"add_dipoles_run/h5/{lattice}/n{n}-H{H}.h5")
vtk_dir = os.path.join(DATA_DIR, f"add_dipoles_run/vtk/{lattice}/n{n}-H{H}")
os.makedirs(vtk_dir, exist_ok=True)

with h5py.File(h5_path, "r") as h5_file:
    write_vtk_box(h5_file, path_target=os.path.join(vtk_dir, "box_l.vtk"))
    for step in range(0, 31 + 1):
        vtk_path = os.path.join(vtk_dir, f"t_{step}.vtk")
        write_vtk_frame_modern(h5_file, path_target=vtk_path, frame=step)